In [ ]:
# Cell 1: Setup

import sys
from pathlib import Path

# Add backend to path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bufferiq.ml.optimization import (
    GridSearchOptimizer,
    RandomSearchOptimizer,
    BayesianOptimizer,
    SearchSpaceRegistry,
    OptimizationPipeline,
    OptimizationResultTracker,
    OptimizationConfig,
)
from bufferiq.ml.trainers.xgboost_trainer import XGBoostTrainer

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Cell 2: Generate Sample Data

np.random.seed(42)
n_samples = 1000
n_features = 50

X = np.random.randn(n_samples, n_features)
y = np.random.randn(n_samples)

split_idx = int(0.8 * n_samples)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

In [ ]:
# Cell 3: Grid Search Setup

trainer = XGBoostTrainer(random_state=42)

param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'n_estimators': [100, 200],
}

grid_optimizer = GridSearchOptimizer(
    model=trainer.model,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    verbose=1,
)

print(f"Total combinations: {grid_optimizer.total_combinations}")

In [ ]:
# Cell 4: Run Grid Search

grid_results = grid_optimizer.search(X_train, y_train)

print(f"\nBest score: {grid_results['best_score']:.4f}")
print(f"Best params: {grid_results['best_params']}")

In [ ]:
# Cell 5: Visualize Grid Search Results

cv_results = grid_results['cv_results']
params_df = pd.DataFrame(cv_results['params'])
params_df['mean_test_score'] = cv_results['mean_test_score']
params_df['mean_train_score'] = cv_results['mean_train_score']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lr_scores = params_df.groupby('learning_rate')['mean_test_score'].mean()
axes[0].plot(lr_scores.index, lr_scores.values, marker='o')
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Mean Test Score (R²)')
axes[0].set_title('Learning Rate vs Performance')
axes[0].grid(True, alpha=0.3)

depth_scores = params_df.groupby('max_depth')['mean_test_score'].mean()
axes[1].plot(depth_scores.index, depth_scores.values, marker='o', color='green')
axes[1].set_xlabel('Max Depth')
axes[1].set_ylabel('Mean Test Score (R²)')
axes[1].set_title('Max Depth vs Performance')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Random Search

from scipy.stats import loguniform, randint

random_space = SearchSpaceRegistry.get_search_space('xgboost', 'random')

random_optimizer = RandomSearchOptimizer(
    model=trainer.model,
    param_distributions=random_space,
    n_iter=20,
    cv=3,
    scoring='r2',
    random_state=42,
    verbose=1,
)

random_results = random_optimizer.search(X_train, y_train)

print(f"\nBest score: {random_results['best_score']:.4f}")
print(f"Best params: {random_results['best_params']}")

In [ ]:
# Cell 7: Compare Optimization Strategies

comparison = pd.DataFrame({
    'Strategy': ['Grid Search', 'Random Search'],
    'Best Score': [
        grid_results['best_score'],
        random_results['best_score'],
    ],
    'Trials': [
        grid_results['total_trials'],
        random_results['total_trials'],
    ],
})

print(comparison)

In [ ]:
# Cell 8: Visualize Strategy Comparison

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(comparison))
ax.bar(x, comparison['Best Score'], color=['blue', 'orange'])
ax.set_xticks(x)
ax.set_xticklabels(comparison['Strategy'])
ax.set_ylabel('Best R² Score')
ax.set_title('Optimization Strategy Comparison')
ax.grid(True, alpha=0.3, axis='y')

for i, v in enumerate(comparison['Best Score']):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Export Best Parameters

import yaml

best_params = {
    'model_type': 'xgboost',
    'best_params': random_results['best_params'],
    'best_score': float(random_results['best_score']),
    'total_trials': random_results['total_trials'],
}

output_path = Path('../outputs/optimizations/notebook_best_params.yaml')
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    yaml.dump(best_params, f, default_flow_style=False)

print(f"Best parameters exported to: {output_path}")

In [ ]:
# Cell 10: Search Space Exploration

random_cv_results = random_results['cv_results']
random_params_df = pd.DataFrame(random_cv_results['params'])
random_params_df['score'] = random_cv_results['mean_test_score']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(random_params_df['learning_rate'], random_params_df['score'], alpha=0.6)
axes[0, 0].set_xlabel('Learning Rate')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Learning Rate vs Score')
axes[0, 0].set_xscale('log')

axes[0, 1].scatter(random_params_df['max_depth'], random_params_df['score'], alpha=0.6, color='green')
axes[0, 1].set_title('Max Depth vs Score')

axes[1, 0].scatter(random_params_df['n_estimators'], random_params_df['score'], alpha=0.6, color='orange')
axes[1, 0].set_title('N Estimators vs Score')

axes[1, 1].scatter(random_params_df['subsample'], random_params_df['score'], alpha=0.6, color='red')
axes[1, 1].set_title('Subsample vs Score')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: Optimization Progress

random_scores = random_cv_results['mean_test_score']
best_scores_so_far = np.maximum.accumulate(random_scores)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(random_scores) + 1), random_scores, 'o-', alpha=0.5, label='Trial Score')
plt.plot(range(1, len(best_scores_so_far) + 1), best_scores_so_far, 'r-', linewidth=2, label='Best So Far')

plt.xlabel('Trial Number')
plt.ylabel('R² Score')
plt.title('Optimization Progress - Random Search')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best score after 5 trials: {best_scores_so_far[4]:.4f}")
print(f"Best score after 10 trials: {best_scores_so_far[9]:.4f}")
print(f"Best score after 20 trials: {best_scores_so_far[-1]:.4f}")

In [ ]:
# Cell 12: Retrain with Best Parameters

best_trainer = XGBoostTrainer(random_state=42)
best_trainer.build_model(random_results['best_params'])

train_metrics = best_trainer.train(
    X_train, y_train,
    X_test, y_test
)

print("\nBest Model Performance:")
print(f"Train R²: {train_metrics['train_r2']:.4f}")
print(f"Test R²: {train_metrics['test_r2']:.4f}")

In [ ]:
# Cell 13: Summary

print("""
Key Takeaways:

1. Grid Search: Exhaustive but slow
2. Random Search: Efficient and scalable
3. Bayesian Optimization: Best for expensive models

Next Steps:
- Use Bayesian optimization
- Add early stopping
- Combine with ensembling
""")